In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from show import *
from utils import *
import pandas as pd
from datetime import datetime
import fnmatch


qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [2]:
def make_map(files, dlon=1, dsine=1e-2, mu_thr=0.1, binning=1):
    grid = np.mgrid[-1:1 + dsine / 2:dsine, :360:dlon]
    grid[0] = np.arcsin(grid[0]) * 180 / np.pi

    mean, variance, coverage, coverage2 = np.zeros_like(grid[0]), np.zeros_like(grid[0]), np.zeros_like(grid[0]), np.zeros_like(grid[0])

    for file in files[:]:
        print(file)

        with fits.open(file) as hdul:
            header = hdul[1].header.copy()
            data = hdul[1].data.copy()

        data = rebin(data, binning, update_header=header)
        view = View.from_header(header)
        transform = view.to_synoptic(correct_mu=True, correct_dr=True, mu_thr=mu_thr) + ToSpherical()

        grid_, alpha = (~transform)(grid)
        data = interp2d(data, *grid_) * alpha

        n = (~np.isnan(data)) * (1 / alpha) ** 4

        coverage += np.nan_to_num(n)
        coverage2 += np.nan_to_num(n ** 2)
        mean_ = mean.copy()
        mean += np.nan_to_num((data - mean) * n / coverage)
        variance += np.nan_to_num((data - mean) * (data - mean_) * n)

    variance = variance / coverage
    mean[coverage == 0] = np.nan
    coverage[coverage == 0] = np.nan
    coverage2[coverage2 == 0] = np.nan
    variance[coverage == 0] = np.nan

    return mean, variance, coverage, coverage2

In [4]:
files = sorted(glob.glob('/home/ulyanov/data/sdo/hmi/2302/*'))
print(len(files))
print(files[0], files[-1])

316
/home/ulyanov/data/sdo/hmi/2302/hmi.M_720s.20250909_060000_TAI.3.magnetogram.fits /home/ulyanov/data/sdo/hmi/2302/hmi.M_720s.20251006_100000_TAI.3.magnetogram.fits


In [5]:
mean, variance, coverage, coverage2 = make_map(files, dsine=1 / 719.5, dlon=0.1)

/home/ulyanov/data/sdo/hmi/2302/hmi.M_720s.20250909_060000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/2302/hmi.M_720s.20250909_080000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/2302/hmi.M_720s.20250909_100000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/2302/hmi.M_720s.20250909_120000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/2302/hmi.M_720s.20250909_140000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/2302/hmi.M_720s.20250909_160000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/2302/hmi.M_720s.20250909_180000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/2302/hmi.M_720s.20250909_200000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/2302/hmi.M_720s.20250909_220000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/2302/hmi.M_720s.20250910_000000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/2302/hmi.M_720s.20250910_020000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/2302/hmi.M_720s.20250910_040000_TAI.3.magnetogram.fits
/home/ulyanov/da

/tmp/ipykernel_59814/3156323304.py:29: RuntimeWarning: invalid value encountered in divide
  variance = variance / coverage


In [6]:
#np.savez('2302hmi.npz', mean=mean, variance=variance, coverage=coverage, coverage2=coverage2)

In [7]:
plt.figure(figsize=(14,8))
plt.imshow(mean, aspect='auto', origin='lower', vmin=-1000, vmax=1000, cmap='hmimag', interpolation='bicubic')
plt.tight_layout()

In [58]:
variance_ = variance * coverage2 / coverage ** 2
mean_ = mean.copy()
mean_[np.abs(mean) < 3 * np.sqrt(variance_)] = np.nan

plt.figure(figsize=(14,8))
plt.imshow(mean_, aspect='auto', origin='lower', vmin=-1000, vmax=1000, cmap='hmimag', interpolation='bicubic')
plt.tight_layout()

/tmp/ipykernel_59814/430584154.py:3: RuntimeWarning: invalid value encountered in sqrt
  mean_[np.abs(mean) < 3 * np.sqrt(variance_)] = np.nan


In [17]:
plt.figure(figsize=(14,8))
plt.imshow(np.sqrt(variance_), aspect='auto', origin='lower', vmin=0, vmax=10, cmap='turbo', interpolation='bicubic')
plt.tight_layout()

/tmp/ipykernel_59814/4082956583.py:2: RuntimeWarning: invalid value encountered in sqrt
  plt.imshow(np.sqrt(variance_), aspect='auto', origin='lower', vmin=0, vmax=10, cmap='turbo', interpolation='bicubic')


In [18]:
files = sorted(glob.glob('/home/ulyanov/data/sdo/hmi/synoptic_maps/*'))

with fits.open(files[0]) as hdul:
    header = hdul[0].header.copy()
    data = hdul[0].data.copy()

In [30]:
plt.figure(figsize=(14,8))
plt.imshow(data, aspect='auto', origin='lower', vmin=-1000, vmax=1000, cmap='hmimag', interpolation='bicubic')
plt.tight_layout()

In [26]:
plt.figure(figsize=(14,8))
plt.imshow(np.sqrt(variance), aspect='auto', origin='lower', vmin=0, vmax=40, cmap='turbo', interpolation='bicubic')
plt.tight_layout()

/tmp/ipykernel_59814/1471131667.py:2: RuntimeWarning: invalid value encountered in sqrt
  plt.imshow(np.sqrt(variance), aspect='auto', origin='lower', vmin=0, vmax=40, cmap='turbo', interpolation='bicubic')


In [64]:
data_ = data.copy()
data_[np.abs(data) < 3 * np.sqrt(variance)] = np.nan

plt.figure(figsize=(14,8))
plt.imshow(data_, aspect='auto', origin='lower', vmin=-1000, vmax=1000, cmap='hmimag', interpolation='bicubic')
plt.tight_layout()

/tmp/ipykernel_59814/862029177.py:2: RuntimeWarning: invalid value encountered in sqrt
  data_[np.abs(data) < 3 * np.sqrt(variance)] = np.nan


In [1]:
x = np.arcsin(np.arange(-1, 1 + 1 / 719.5 / 2, 1 / 719.5)) * 180 / np.pi

plt.figure(figsize=(10,8))
plt.plot(x, np.nanmean(data, axis=1))
plt.plot(x, np.nanmean(mean, axis=1))
#plt.plot(x, np.nanmean(mean_, axis=1))

plt.xlim(-90,90)
plt.ylim(-10,10)

plt.grid(True)
plt.tight_layout()

NameError: name 'np' is not defined